# CS781 Phase 4 - Court Judgment Prediction - notebook `2`

**Base design:** case text (line breaks removed) in Phase-1 tokenization -> the **last 1,024 tokens as 2 chunks of 512**
-> Phase-1 backbone -> mean-pooled chunk vectors + relative chunk position (16 sinusoids) -> **attention pooling** over the
chunks -> 2 labels; cross-entropy with balanced class weights; 1 epoch over train.json, batches of 8 cases; dev
macro-F1 evaluated every 25% of the steps, best checkpoint kept. Training cases that also occur in dev/test are removed.
**Techniques** (T, every model): LoRMA (compulsory), AdaLoRA, adapters, BitFit, gradual unfreezing, partial FT.
**Representation study** (R, models B and D, LoRMA, the same 8,000-case subset): which part of the judgment, how much, pooling,
class weights. **Extras** (X, run last, skipped if the session runs out of time): full FT, LoRA.

| ID | GPU | run | technique | change vs base design | group | estimate |
|---|---|---|---|---|---|---|
| D04 | GPU0 | `D04_bitfit` | BitFit (norm gains) | - | T | ~58 min |
| D06 | GPU0 | `D06_partial` | Partial FT (top 4) | - | T | ~46 min |
| B03 | GPU0 | `B03_adapter` | Houlsby adapter | - | T | ~45 min |
| B05 | GPU0 | `B05_gradual` | Gradual unfreezing | - | T | ~36 min |
| B06 | GPU0 | `B06_partial` | Partial FT (top 4) | - | T | ~31 min |
| B13 | GPU0 | `B13_rep_last4` | LoRMA r16 | last 2,048 tokens (4 chunks) | R | ~22 min |
| B14 | GPU0 | `B14_rep_first2` | LoRMA r16 | first 1,024 tokens | R | ~11 min |
| B17 | GPU0 | `B17_rep_meanpool` | LoRMA r16 | mean pooling of chunks | R | ~11 min |
| B12 | GPU0 | `B12_rep_last1` | LoRMA r16 | last 512 tokens only | R | ~7 min |
| B08 | GPU0 | `B08_lora` | LoRA r16 | - | X | ~45 min |
| D05 | GPU1 | `D05_gradual` | Gradual unfreezing | - | T | ~52 min |
| B02 | GPU1 | `B02_adalora` | AdaLoRA r12->8 | - | T | ~50 min |
| B01 | GPU1 | `B01_lorma` | LoRMA r16 | - | T | ~45 min |
| B04 | GPU1 | `B04_bitfit` | BitFit (norm gains) | - | T | ~40 min |
| B16 | GPU1 | `B16_rep_spread8` | LoRMA r16 | whole document: 8 chunks spread evenly | R | ~45 min |
| B11 | GPU1 | `B11_rep_ref` | LoRMA r16 | 8k-doc subset (reference) | R | ~11 min |
| B15 | GPU1 | `B15_rep_firstlast` | LoRMA r16 | first 512 + last 512 | R | ~11 min |
| B18 | GPU1 | `B18_rep_noweights` | LoRMA r16 | no class weights | R | ~11 min |
| B07 | GPU1 | `B07_full` | Full FT | - | X | ~52 min |

**Inputs:** dataset `cs781-p4-code`, `cs781-legal-data` (tokenizer), Phase-1 checkpoint(s) of model B + D.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~5.3 h of runs (GPU0 5.2 h, GPU1 5.3 h) after ~5 min of tokenization and a
~5 min smoke test. Each run has a time limit (it then stops, is evaluated on dev and kept); no run starts after
6.5 h and every run is cut to end by then, so the notebook finishes within ~7 h (deadline day). A run that fails the smoke test is dropped
(its error is in the report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt`, `bundle_p4_2.zip` (dev/test probabilities,
attention weights, logs; input of `cs781-p4-final`), `REPORT_p4_2.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
NB_START = time.time()
W = "/kaggle/working"
WORK, P4DATA, RUNS, MODELS = f"{W}/p4code", "/tmp/p4data", f"{W}/runs", f"{W}/models"
CACHE = "/tmp/p4_cache.npz"                        # tokenized data (outside /kaggle/working: not part of the output)
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions", P4DATA):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("cjpe_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-4 data from Hugging Face (Phase_4_CJPE/Train/{train,dev}.json, Phase_4_CJPE/Test/test.json)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
files = [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_4")]
print("HF files:", files)
for f in files:
    dst = os.path.join(P4DATA, f)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- unit tests, then tokenize every split once (shared by all runs; several CPU processes, a few minutes)
r = subprocess.run("python tests_cjpe.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"
import cjpe_data
t0 = time.time()
INFO = cjpe_data.build_cache(P4DATA, TOK_ROOT, CACHE)
print(f"cache built in {time.time() - t0:.0f}s")
if "test" not in INFO:
    print("WARNING: no test.json found - runs will train and save models, but write no submission files")

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'cjpe_data.py', 'cjpe_eval.py', 'cjpe_model.py', 'cjpe_train.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'tests_cjpe.py']
HF files: ['Phase_4_CJPE/Test/test.json', 'Phase_4_CJPE/Train/dev.json', 'Phase_4_CJPE/Train/train.json']


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

  model b: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
  model d: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

ok test_chunks
ok test_batch
ok test_metric
ok test_files
ok test_cache
ALL TESTS PASSED
 
train: {"rows_in_file": 37387, "unique_ids": 32305, "docs": 32305, "dropped_overlap_with_dev_test": 0, "tokens": 116612264, "tok_median": 2308, "tok_p90": 7371, "tok_max": 476066, "label_1": 13385, "label_0": 18920, "seconds": 181}
dev: {"rows_in_file": 1988, "unique_ids": 994, "docs": 994, "tokens": 4372529, "tok_median": 3302, "tok_p90": 8055, "tok_max": 53526, "label_1": 497, "label_0": 497, "seconds": 8}
test: {"rows_in_file": 1517, "unique_ids": 1517, "docs": 1517, "tokens": 6835239, "tok_median": 3441, "tok_p90": 8420, "tok_max": 50399, "label_1": 762, "label_0": 755, "seco

In [2]:
NB_HOURS = 6.5                                     # deadline day: no new run starts after 6.5 h; runs are cut to fit

def cmd(e, out, extra, model_dir):
    return (f"python cjpe_train.py --ckpt {CKPT[e['model']]} --tok_root {TOK_ROOT} --data_dir {P4DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {e['args']} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: a few documents, a few steps."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            if smoke:
                extra = "--max_docs 24 --batch_docs 4 --eval_every 0.5" + ("" if e is jobs[0] else " --no_test")
            else:
                left = NB_HOURS * 60 - (time.time() - NB_START) / 60
                if left < 20:
                    print(f"[GPU{gpu}] {t}: SKIPPED (notebook time budget used up)", flush=True)
                    status[t] = "skipped"
                    continue
                extra = f"--max_minutes {min(e['max_min'], left - 15):.0f}"
            status[t] = run(t, cmd(e, f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Every experiment once on 24 documents. A failing experiment is dropped (its error is printed and reported)
    instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = '2'
QUEUES = [[{'id': 'D04',
   'tag': 'D04_bitfit',
   'model': 'd',
   'tech': 'BitFit (norm gains)',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method norm',
   'max_min': 99},
  {'id': 'D06',
   'tag': 'D06_partial',
   'model': 'd',
   'tech': 'Partial FT (top 4)',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method partial --top_k 4',
   'max_min': 78},
  {'id': 'B03',
   'tag': 'B03_adapter',
   'model': 'b',
   'tech': 'Houlsby adapter',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method adapter --bottleneck 64',
   'max_min': 77},
  {'id': 'B05',
   'tag': 'B05_gradual',
   'model': 'b',
   'tech': 'Gradual unfreezing',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method gradual',
   'max_min': 63},
  {'id': 'B06',
   'tag': 'B06_partial',
   'model': 'b',
   'tech': 'Partial FT (top 4)',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method partial --top_k 4',
   'max_min': 55},
  {'id': 'B13',
   'tag': 'B13_rep_last4',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'last 2,048 tokens (4 chunks)',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --n_chunks 4',
   'max_min': 41},
  {'id': 'B14',
   'tag': 'B14_rep_first2',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'first 1,024 tokens',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --select first',
   'max_min': 23},
  {'id': 'B17',
   'tag': 'B17_rep_meanpool',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'mean pooling of chunks',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --pool mean',
   'max_min': 23},
  {'id': 'B12',
   'tag': 'B12_rep_last1',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'last 512 tokens only',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --n_chunks 1',
   'max_min': 16},
  {'id': 'B08',
   'tag': 'B08_lora',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method lora --rank 16 --alpha 32',
   'max_min': 77}],
 [{'id': 'D05',
   'tag': 'D05_gradual',
   'model': 'd',
   'tech': 'Gradual unfreezing',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method gradual',
   'max_min': 88},
  {'id': 'B02',
   'tag': 'B02_adalora',
   'model': 'b',
   'tech': 'AdaLoRA r12->8',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method adalora --rank 12 --alpha 24 --target_rank 8',
   'max_min': 84},
  {'id': 'B01',
   'tag': 'B01_lorma',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4',
   'max_min': 77},
  {'id': 'B04',
   'tag': 'B04_bitfit',
   'model': 'b',
   'tech': 'BitFit (norm gains)',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method norm',
   'max_min': 70},
  {'id': 'B16',
   'tag': 'B16_rep_spread8',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'whole document: 8 chunks spread evenly',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --select spread --n_chunks 8',
   'max_min': 77},
  {'id': 'B11',
   'tag': 'B11_rep_ref',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': '8k-doc subset (reference)',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000',
   'max_min': 23},
  {'id': 'B15',
   'tag': 'B15_rep_firstlast',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'first 512 + last 512',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --select firstlast',
   'max_min': 23},
  {'id': 'B18',
   'tag': 'B18_rep_noweights',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'no class weights',
   'args': '--select last --n_chunks 2 --pool attn --method lorma --rank 16 --alpha 32 --lr 2e-4 --max_train_docs 8000 --class_weight none',
   'max_min': 23},
  {'id': 'B07',
   'tag': 'B07_full',
   'model': 'b',
   'tech': 'Full FT',
   'change': '-',
   'args': '--select last --n_chunks 2 --pool attn --method full',
   'max_min': 88}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU0] smoke_D04_bitfit: exit 0 | [05:05:21] TEST: 24 cases, 15 predicted 1 -> submission.zip / [05:05:21] DONE
[GPU1] smoke_D05_gradual: exit 0 | [05:05:22] TEST: 24 cases, 10 predicted 1 -> submission.zip / [05:05:22] DONE
[GPU0] smoke_D06_partial: exit 0 | [05:05:37] TEST: no test.json - no submission written / [05:05:37] DONE
[GPU1] smoke_B02_adalora: exit 0 | [05:05:38] TEST: no test.json - no submission written / [05:05:38] DONE
[GPU0] smoke_B03_adapter: exit 0 | [05:05:48] TEST: no test.json - no submission written / [05:05:48] DONE
[GPU1] smoke_B01_lorma: exit 0 | [05:05:50] TEST: no test.json - no submission written / [05:05:50] DONE
[GPU0] smoke_B05_gradual: exit 0 | [05:05:59] TEST: no test.json - no submission written / [05:05:59] DONE
[GPU1] smoke_B04_bitfit: exit 0 | [05:06:01] TEST: no test.json - no submission written / [05:06:01] DONE
[GPU0] smoke_B06_partial: exit 0 | [05:06:10] TEST: no test.json - no submission written / [05:06:10] DONE
[GPU1] smoke_B16_rep_spread8:

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU1] D05_gradual: exit 0 | [06:08:07] TEST: 1517 cases, 620 predicted 1 -> submission.zip / [06:08:07] DONE
[GPU0] D04_bitfit: exit 0 | [06:16:32] TEST: 1517 cases, 833 predicted 1 -> submission.zip / [06:16:32] DONE
[GPU0] D06_partial: exit 0 | [06:59:59] TEST: 1517 cases, 561 predicted 1 -> submission.zip / [06:59:59] DONE
[GPU1] B02_adalora: exit 0 | [07:06:43] TEST: 1517 cases, 603 predicted 1 -> submission.zip / [07:06:43] DONE
[GPU0] B03_adapter: exit 0 | [07:43:37] TEST: 1517 cases, 574 predicted 1 -> submission.zip / [07:43:37] DONE
[GPU1] B01_lorma: exit 0 | [08:00:28] TEST: 1517 cases, 491 predicted 1 -> submission.zip / [08:00:28] DONE
[GPU0] B05_gradual: exit 0 | [08:24:49] TEST: 1517 cases, 610 predicted 1 -> submission.zip / [08:24:49] DONE
[GPU1] B04_bitfit: exit 0 | [08:44:25] TEST: 1517 cases, 836 predicted 1 -> submission.zip / [08:44:25] DONE
[GPU0] B06_partial: exit 0 | [08:55:37] TEST: 1517 cases, 597 predicted 1 -> submission.zip / [08:55:37] DONE
[GPU0] B13_rep

## 3. Report

In [6]:
def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                        | technique            | change                         | dev F1 | AUC    | F1@thr      | P      | R      | pred1 | steps     | trainable | min  | tok/s  | mem   | ms/doc"]
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SKIPPED/DROPPED" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:26s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]
        L.append(f"{e['id']} | {e['tag']:26s} | {e['tech'][:20]:20s} | {e['change'][:30]:30s} | {dv['macro_f1']:.4f} | "
                 f"{dv['auc']:.4f} | {dv['macro_f1_best_threshold']:.4f}@{dv['best_threshold']:.2f} | {dv['macro_p']:.4f} | "
                 f"{dv['macro_r']:.4f} | {100 * dv['pred_1_share']:4.0f}% | {s.get('best_step', 0):4d}/{s.get('steps_planned', 0):4d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | "
                 f"{s.get('peak_mem_gb') or 0:4.1f}G | {s.get('infer_ms_per_doc', 0):6.1f}")
        if os.path.exists(f"{RUNS}/{e['tag']}/submission.zip"):
            shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    bundle = f"{W}/bundle_p4_{name}.zip"              # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
            if os.path.exists(f"{W}/logs/{e['tag']}.txt"):
                z.write(f"{W}/logs/{e['tag']}.txt", arcname=f"{e['tag']}/stdout.txt")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_p4_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT 2 ===
ID  | run                        | technique            | change                         | dev F1 | AUC    | F1@thr      | P      | R      | pred1 | steps     | trainable | min  | tok/s  | mem   | ms/doc
D04 | D04_bitfit                 | BitFit (norm gains)  | -                              | 0.6050 | 0.6440 | 0.6127@0.52 | 0.6085 | 0.6066 |   57% | 3030/4039 |     0.22M |   68 |  7,460 |  5.2G |   41.6
D06 | D06_partial                | Partial FT (top 4)   | -                              | 0.7317 | 0.8315 | 0.7453@0.35 | 0.7595 | 0.7374 |   35% | 4039/4039 |    30.08M |   42 | 12,428 |  2.6G |   41.7
B03 | B03_adapter                | Houlsby adapter      | -                              | 0.7054 | 0.8022 | 0.7306@0.41 | 0.7342 | 0.7123 |   35% | 4039/4039 |     2.58M |   42 | 12,060 |  3.1G |   33.4
B05 | B05_gradual                | Gradual unfreezing   | -                              | 0.7476 | 0.8490 | 0.7603@0.35 | 0.7679 | 0.7515 |   38% | 4039/4039 |    85.